# Current BERT versus LLM VAD

Exploratory comparison of **complete saved texts**. No segmentation, new rewrites, or changes to STDI. The initial pilot uses 20 synthetic context pairs and 30 original/final simulation pairs (the same five news items across six chains), deduplicated to 75 texts.

Each LLM input contains only the text. The frozen rubric uses nominal 1–5 scores for expressed affective tone and narrator/speaker agency for dominance. BERT and LLM are comparators; neither is a human gold standard. Larger drift does not establish better accuracy.


In [ ]:
import json
import subprocess
import sys
from pathlib import Path

import pandas as pd
import plotly.express as px
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "pyproject.toml").exists()
OUTPUT_DIR = PROJECT_ROOT / "output/audit/LLMVADComparison_20261006"

# Set True explicitly to run/resume external API evaluations.
RUN_EVALUATION = False
if RUN_EVALUATION:
    subprocess.run(
        [
            sys.executable,
            str(PROJECT_ROOT / "scripts/compare_llm_vad.py"),
            "--model",
            "gpt-6-luna",
            "--provider",
            "chatgpt",
            "--output-dir",
            str(OUTPUT_DIR),
            "--stage",
            "all",
        ],
        cwd=PROJECT_ROOT,
        check=True,
    )

## Frozen inputs, scale, and missing values

Inspect `configuration.json` for the exact rubric, model, provider, sampling rule, and source hashes. Signed deltas are compared minus reference: **positive minus negative** for context pairs, and **final minus original** for simulations. Drift is the mean of `min(abs(delta)/4, 1)` across V, A, D, matching the current theoretical amplitude normalization. There is no sample min/max scaling.

Missing scores remain missing. Joint denominators are metric-specific. Context polarity supplies a valence direction control, not an assumed arousal or dominance direction. Simulation pairs share originals; 30 pairs represent five news items, not 30 independent items.


In [ ]:
configuration = json.loads((OUTPUT_DIR / "configuration.json").read_text(encoding="utf-8"))
manifest = json.loads((OUTPUT_DIR / "manifest.json").read_text(encoding="utf-8"))
scores = pd.read_csv(OUTPUT_DIR / "text_scores.csv")
pairs = pd.read_csv(OUTPUT_DIR / "pair_comparison.csv")
summary = pd.read_csv(OUTPUT_DIR / "pair_summary.csv")
agreement = pd.read_csv(OUTPUT_DIR / "score_agreement.csv")
display(manifest)
print(configuration["rubric"])
display(summary)
display(agreement)

In [ ]:
fig = px.scatter(
    pairs,
    x="bert_vad_drift",
    y="llm_vad_drift",
    color="dataset",
    hover_data=["pair_id", "chain_code", "news_id"],
    title="Full-text VAD drift: BERT versus LLM",
    labels={"bert_vad_drift": "Current BERT drift (0–1)", "llm_vad_drift": "LLM drift (0–1)"},
)
upper = max(pairs[["bert_vad_drift", "llm_vad_drift"]].max().max() * 1.1, 0.05)
fig.add_shape(type="line", x0=0, y0=0, x1=upper, y1=upper, line=dict(color="gray", dash="dash"))
fig.update_xaxes(range=[0, upper])
fig.update_yaxes(range=[0, upper])
fig.show()

In [ ]:
dimension_rows = pd.concat(
    [
        scores[["text_id", f"bert_{dimension}", f"llm_{dimension}"]]
        .rename(columns={f"bert_{dimension}": "bert", f"llm_{dimension}": "llm"})
        .assign(dimension=dimension)
        for dimension in ("valence", "arousal", "dominance")
    ],
    ignore_index=True,
)
fig = px.scatter(
    dimension_rows,
    x="bert",
    y="llm",
    facet_col="dimension",
    hover_data=["text_id"],
    title="Individual full-text scores (nominal 1–5)",
    labels={"bert": "Current BERT", "llm": "LLM"},
)
fig.update_xaxes(range=[1, 5])
fig.update_yaxes(range=[1, 5])
fig.show()

In [ ]:
largest_gaps = (
    pairs.assign(absolute_gap=pairs.llm_minus_bert_vad_drift.abs())
    .sort_values("absolute_gap", ascending=False)
    .head(5)
)
for pair in largest_gaps.itertuples():
    print(f"\n{pair.pair_id}: BERT={pair.bert_vad_drift:.4f}; LLM={pair.llm_vad_drift:.4f}")
    print("REFERENCE:", pair.reference_text)
    print("COMPARED:", pair.compared_text)
    selected = scores[scores.text_id.isin([pair.reference_id, pair.compared_id])]
    display(selected.drop(columns=["text"]).T)

## Interpretation and reproducibility

Inspect rationales, literal evidence, and ambiguities before interpreting gaps. Dominance measures conveyed emotional agency in this rubric, not certainty or institutional authority. The BERT model card does not establish an identical annotation perspective. Equal scale amplitudes do not calibrate the models.

The LLM receives the full text; BERT uses its existing 512 WordPiece limit. Inspect truncation flags. This pilot uses one successful LLM assessment per unique text; validation retries are not independent replicates. It cannot assess repeatability, factual veracity, actual reader impact, Portuguese validity, or superiority without independent human annotations.

To reproduce or resume: `uv run python scripts/compare_llm_vad.py --stage all`. To regenerate tables from saved valid responses: `uv run python scripts/compare_llm_vad.py --stage report`. Changed model, rubric, or inputs require a new output directory. Raw responses and validation attempts are retained in `llm_responses/`.
